# Post 3 — Deployment (§10, §5.2, §5.3)

Throughput + 2.5M-forecast SLA simulation, cold-start (MAE vs history length), and drift A/B (post-earthquake degradation) — all for LightGBM. TabPFN-TS legs are **pending**; the artifacts written here (`runtime_log.parquet`, `throughput.parquet`, `coldstart.parquet`, `drift.parquet`) already carry an `approach`/`model_name` column so the TabPFN rows append cleanly.

## 0. Control panel

In [ ]:
# ---- Control panel (deployment §10) ---------------------------------------
CONFIG_NAME = "default.yaml"
OUTPUT_SUBDIR = "deployment"
FEATURE_SET = "full"                 # deployment experiments use the practitioner set
USE_CORE_SLICE = False               # throughput wants the full panel; see knobs below
THROUGHPUT_ON_FULL_PANEL = True      # time a single-origin full-panel predict (§10.1)
COLDSTART_MAX_PAIRS = 20             # subsample held-out pairs to keep wall-clock sane
COLDSTART_FEATURE_SET = "lag"        # cheaper than "full" across many pair x history calls
RUN_DRIFT = True                     # §5.2 / §10.4 variants A and B
QUICK = True
RANDOM_SEED = 42

## 1. Imports + helpers

In [ ]:
import sys, time, json, copy, platform, subprocess
from pathlib import Path

# Make `src` importable whether the notebook runs from notebooks/ or repo root.
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "src" / "demand").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.demand.config import load_config, resolve_path
from src.demand.data.load import load_raw_data, build_panel
from src.demand.data.splits import (
    build_main_splits, build_core_slice, family_regime, regime_horizon, regime_costs,
)
from src.demand.data.supervised_frame import build_supervised_frame
from src.demand.models import lgbm_point, lgbm_quantile
from src.demand.models.lgbm_base import train_lgbm

def git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"],
                                       stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return "no-git"

In [ ]:
def point_params(cfg):
    p = dict(cfg["lgbm"]["defaults"])
    p.update(cfg["lgbm"]["objectives"]["point"])
    return p

def train_point(panel_, feature_set, train_origins, val_origins=None,
                     series_filter=None, include_earthquake=True):
    """Build the supervised frame from `panel_` and fit one point (log1p+L2) booster."""
    frame_train = build_supervised_frame(
        panel_, origins=train_origins, horizon=cfg["horizon_days"],
        feature_set=feature_set, config=cfg, mode="train",
        holidays=raw.holidays, stores=raw.stores, series_filter=series_filter,
        include_earthquake=include_earthquake,
    )
    frame_val = None
    if val_origins:
        frame_val = build_supervised_frame(
            panel_, origins=val_origins, horizon=cfg["horizon_days"],
            feature_set=feature_set, config=cfg, mode="train",
            holidays=raw.holidays, stores=raw.stores, series_filter=series_filter,
            include_earthquake=include_earthquake,
        )
    params = point_params(cfg)
    n_est = params.pop("n_estimators", 2000)
    if QUICK:
        n_est = min(n_est, 400)
    return train_lgbm(
        frame_train=frame_train, frame_val=frame_val, feature_set=feature_set,
        params=params, n_estimators=n_est,
        early_stopping_rounds=cfg["lgbm"]["search_space"]["early_stopping_rounds"],
        include_earthquake=include_earthquake, name=f"lgbm_point_{feature_set}",
    )

## 2. Load data + scope

In [ ]:
cfg = copy.deepcopy(load_config(CONFIG_NAME))
cfg["seed"] = RANDOM_SEED
splits = build_main_splits(cfg)

t0 = time.time()
raw = load_raw_data(cfg)
panel = build_panel(raw, include_test=False)
print(f"loaded panel in {time.time()-t0:.1f}s | rows={len(panel):,} | "
      f"series={panel[['store_nbr','family']].drop_duplicates().shape[0]:,}")

if USE_CORE_SLICE:
    series_filter = build_core_slice(panel, raw.stores, cfg, write_csv=False)
    print(f"core slice: {len(series_filter)} series, "
          f"{series_filter['family'].nunique()} families")
else:
    series_filter = None
    print("scope: full panel")

results_dir = resolve_path(cfg, "results_dir")
figures_dir = resolve_path(cfg, "figures_dir")
out_dir = results_dir / OUTPUT_SUBDIR
out_dir.mkdir(parents=True, exist_ok=True)
print(f"git commit: {git_commit()}  ->  output: {out_dir}")

## 3. Throughput + weekly-refresh SLA (§10.1)

In [ ]:
from src.demand.deployment.throughput import (
    time_forecast, append_runtime_log, simulate_weekly_refresh, RuntimeRow,
)

# §10.1 — time one full-origin LightGBM forecast pass and log it.
origin = splits.test_origins[3]      # middle origin
sf = None if THROUGHPUT_ON_FULL_PANEL else build_core_slice(panel, raw.stores, cfg, write_csv=False)
train_origins = list(splits.training_origins)
art = train_point(panel, FEATURE_SET, train_origins, series_filter=sf)
frame_fc = build_supervised_frame(
    panel, origins=[origin], horizon=cfg["horizon_days"], feature_set=FEATURE_SET,
    config=cfg, mode="forecast", holidays=raw.holidays, stores=raw.stores,
    series_filter=sf)
n_series = frame_fc[["store_nbr", "family"]].drop_duplicates().shape[0]

forecasts, row = time_forecast(lambda: lgbm_point.predict(art, frame_fc),
                               model_name="lgbm_point", feature_set=FEATURE_SET,
                               n_series=n_series, origin=str(origin.date()))
append_runtime_log(row, results_dir / "runtime_log.parquet")
print(f"{n_series} series x 28d in {row.wall_clock_seconds:.2f}s | "
      f"{row.predictions_per_second:,.0f} preds/s")

# §10.1.1 — extrapolate to the 2.5M-forecast weekly refresh.
sla = simulate_weekly_refresh(row.predictions_per_second, cfg)
print(json.dumps(sla, indent=2))
pd.DataFrame([sla]).to_parquet(results_dir / "throughput.parquet", index=False)

## 4. Cold-start (§5.3 / §10.3)

In [ ]:
# §5.3 / §10.3 — cold-start: held-out pairs, 7/28/90 days of history, MAE on days 1-7.
from src.demand.deployment.coldstart import run_coldstart, aggregate_coldstart, exclude_pairs
from src.demand.data.splits import build_cold_start_holdout
from src.demand.evaluation import plots

# Training excludes the FULL seeded holdout (no held-out pair ever trains).
holdout_pairs = build_cold_start_holdout(panel, cfg)
training_panel = exclude_pairs(panel, holdout_pairs)
# Pre-train ONCE on the non-held-out panel (held-out pairs never seen in training).
cs_origins = list(splits.training_origins)
cs_artifact = train_point(training_panel, COLDSTART_FEATURE_SET, cs_origins)

def lgbm_coldstart_forecast_fn(truncated_panel, origin, feature_set, **kwargs):
    """Forecast the truncated panel with the pre-trained booster. Accepts
    training_panel/holdout_pairs kwargs (required by the no-leakage guard) but
    ignores them — training already happened on the held-out-free panel."""
    frame = build_supervised_frame(
        truncated_panel, origins=[origin], horizon=cfg["horizon_days"],
        feature_set=feature_set, config=cfg, mode="forecast",
        holidays=raw.holidays, stores=raw.stores)
    if frame.empty:
        return None
    return lgbm_point.predict(cs_artifact, frame)   # has a `pred` column

# run_coldstart rebuilds the same seeded holdout internally; `max_pairs` caps how
# many are evaluated (the slowest experiment: pairs x history lengths x features).
cs_results = run_coldstart(
    panel, cfg, lgbm_coldstart_forecast_fn,
    approach_name="lgbm_point", feature_set=COLDSTART_FEATURE_SET,
    holidays=raw.holidays, stores=raw.stores, max_pairs=COLDSTART_MAX_PAIRS)
cs_agg = aggregate_coldstart(cs_results)
print(cs_agg.to_string(index=False))
cs_results.to_parquet(results_dir / "coldstart.parquet", index=False)
if not cs_agg.empty:
    plots.coldstart_curve(cs_agg, figures_dir / OUTPUT_SUBDIR)

## 5. Drift A/B (§5.2 / §10.4)

In [ ]:
# §5.2 / §10.4 — distribution shift (earthquake). Variant A drops earthquake
# features, Variant B keeps them. The drift driver carves a held-out normal
# window so the degradation ratio compares out-of-sample to out-of-sample.
from src.demand.deployment.drift import run_drift_variant, summarize_drift

DRIFT_FEATURE_SET = "full"   # earthquake features live in the `full` set (§4.3.5)

def _drift_train_origins(train_end):
    s = cfg["splits"]
    grid = pd.date_range(s["training_origins_start"], train_end, freq=s["training_origins_freq"])
    h = cfg["horizon_days"]
    return [pd.Timestamp(o) for o in grid if o + pd.Timedelta(days=h) <= pd.Timestamp(train_end)]

def lgbm_drift_train_fn(panel_train, include_eq):
    origins = _drift_train_origins(panel_train["date"].max())
    return train_point(panel_train, DRIFT_FEATURE_SET, origins,
                            include_earthquake=include_eq)

def lgbm_drift_predict_fn(artifact, forecast_panel):
    # Roll 28-day origins across the evaluation window. Build features from the
    # FULL panel (captured in closure) so lags have history; the window slice
    # only tells us which dates to score.
    wstart, wend = forecast_panel["date"].min(), forecast_panel["date"].max()
    h = cfg["horizon_days"]
    origins, o = [], wstart - pd.Timedelta(days=1)
    while o < wend:
        origins.append(pd.Timestamp(o))
        o = o + pd.Timedelta(days=h)
    frame = build_supervised_frame(
        panel, origins=origins, horizon=h, feature_set=DRIFT_FEATURE_SET,
        config=cfg, mode="forecast", holidays=raw.holidays, stores=raw.stores)
    fc = lgbm_point.predict(artifact, frame)
    return fc[(fc["date"] >= wstart) & (fc["date"] <= wend)]

drift_frames = []
if RUN_DRIFT:
    for variant in ("A", "B"):
        res = run_drift_variant(panel, cfg, variant, lgbm_drift_train_fn,
                                lgbm_drift_predict_fn, "lgbm_point", DRIFT_FEATURE_SET)
        drift_frames.append(res)
    drift_df = pd.concat(drift_frames, ignore_index=True)
    drift_df.to_parquet(results_dir / "drift.parquet", index=False)
    display(summarize_drift(drift_df))

from src.demand.reporting.summary_tables import build_summary_tables
print("summary ->", build_summary_tables(cfg))

## 6. TabPFN-TS legs — _pending_

Reuse `run_coldstart` / `run_drift_variant` with TabPFN adapters once §6.2 exists; for throughput, wrap the per-series TabPFN inference loop in `time_forecast` and `simulate_weekly_refresh` — the operational story of Post 3 (prior P14).